In [ ]:
import os
import pandas as pd


In [ ]:
import kagglehub
path = kagglehub.dataset_download("thedevastator/airbnb-prices-in-european-cities")

Using Colab cache for faster access to the 'airbnb-prices-in-european-cities' dataset.


In [ ]:
os.listdir(path)

['paris_weekends.csv',
 'rome_weekends.csv',
 'berlin_weekdays.csv',
 'amsterdam_weekends.csv',
 'lisbon_weekends.csv',
 'amsterdam_weekdays.csv',
 'rome_weekdays.csv',
 'athens_weekends.csv',
 'budapest_weekends.csv',
 'barcelona_weekdays.csv',
 'barcelona_weekends.csv',
 'budapest_weekdays.csv',
 'vienna_weekdays.csv',
 'london_weekends.csv',
 'berlin_weekends.csv',
 'paris_weekdays.csv',
 'athens_weekdays.csv',
 'vienna_weekends.csv',
 'lisbon_weekdays.csv',
 'london_weekdays.csv']

In [ ]:
cities = []
for file_name in os.listdir(path):
    if file_name.endswith('.csv'):
        cities.append(file_name.split('_')[0].capitalize())
cities

['Paris',
 'Rome',
 'Berlin',
 'Amsterdam',
 'Lisbon',
 'Amsterdam',
 'Rome',
 'Athens',
 'Budapest',
 'Barcelona',
 'Barcelona',
 'Budapest',
 'Vienna',
 'London',
 'Berlin',
 'Paris',
 'Athens',
 'Vienna',
 'Lisbon',
 'London']

In [ ]:
cities =list(set(cities))
cities

['Berlin',
 'Vienna',
 'Budapest',
 'Rome',
 'Amsterdam',
 'Paris',
 'London',
 'Lisbon',
 'Athens',
 'Barcelona']

In [ ]:
%pip install selenium webdriver-manager pandas

# scrapping

In [ ]:
import time
import json
import re
import os
import random
import traceback
import pandas as pd
from urllib.parse import urljoin, quote
from google.colab import files
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException,
    StaleElementReferenceException,
)

In [ ]:
# 1. Install Google Chrome via dedicated apt configuration
print("Downloading and installing stable Chrome & matching Chromedriver...")
os.system("apt-get update")
os.system("apt-get install -y wget curl unzip")
os.system("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb")
os.system("dpkg -i google-chrome-stable_current_amd64.deb || apt-get install -fy")

os.system("google-chrome --version")
# 2. Configure Chrome Options for Headless execution in Colab
options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("--lang=en-US")
options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/131.0.0.0 Safari/537.36"
)
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--lang=en-US")
options.add_argument("--disable-notifications")
options.add_argument("--disable-popup-blocking")
options.add_argument("--disable-extensions")

prefs = {
    "profile.managed_default_content_settings.images": 2,
    "profile.managed_default_content_settings.stylesheets": 2
}

options.add_experimental_option(
    "prefs",
    prefs
)
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 30)


In [ ]:
def safe_text(driver, by, selector, attr=None):
    try:
        el = driver.find_element(by, selector)
        return el.get_attribute(attr) if attr else el.text.strip()
    except NoSuchElementException:
        return None


def close_popups(driver):
    for xp in [
        "//button[@aria-label='Close']",
        "//button[contains(., 'Accept')]",
        "//button[contains(., 'OK')]",
    ]:
        try:
            driver.find_element(By.XPATH, xp).click()
            time.sleep(0.5)
        except Exception:
            pass


def get_json_ld(driver):
    result = {}
    scripts = driver.find_elements(By.CSS_SELECTOR, 'script[type="application/ld+json"]')
    for s in scripts:
        try:
            data = json.loads(s.get_attribute("innerHTML"))
        except Exception:
            continue
        if isinstance(data, dict) and data.get("@type") == "VacationRental":
            result["title"] = data.get("name")
            result["lat"] = data.get("latitude")
            result["lng"] = data.get("longitude")
            agg = data.get("aggregateRating") or {}
            result["rating"] = agg.get("ratingValue")
            result["reviews"] = agg.get("ratingCount")
            break
    return result

def scroll_through_page(driver, steps=8, pause=1.2):
    height = driver.execute_script("return document.body.scrollHeight")
    for i in range(1, steps + 1):
        driver.execute_script(f"window.scrollTo(0, {height * i / steps});")
        time.sleep(pause)
def parse_overview(driver):
    """6 guests · 2 bedrooms · 3 beds · 1 bath  + room_type"""
    out = {"guests": None, "bedrooms": None, "beds": None, "baths": None,
           "room_type": None, "property_type": None}
    try:
        h2 = driver.find_element(
            By.XPATH, "//h2[contains(text(),' in ') and contains(text(),'Entire') or "
                      "contains(text(),'Private room') or contains(text(),'Shared room') or "
                      "contains(text(),'Room in') or contains(text(),'rental unit')]")
        header = h2.text.strip()  # Entire rental unit in Paris, France
        out["property_type"] = header.split(" in ")[0].strip()
        low = header.lower()
        if low.startswith("entire"):
            out["room_type"] = "Entire home/apt"
        elif low.startswith("private"):
            out["room_type"] = "Private room"
        elif low.startswith("shared"):
            out["room_type"] = "Shared room"
        else:
            out["room_type"] = header.split(" in ")[0].strip()

        ol = h2.find_element(By.XPATH, "./following::ol[1]")
        text = ol.text.replace("\n", " ")
        for key, pattern in {
            "guests": r"(\d+)\s+guest",
            "bedrooms": r"(\d+)\s+bedroom",
            "beds": r"(\d+)\s+bed(?!room)",
            "baths": r"(\d+(?:\.\d+)?)\s+(?:private\s+|shared\s+)?bath",
        }.items():
            m = re.search(pattern, text, re.I)
            if m:
                out[key] = float(m.group(1)) if "." in m.group(1) else int(m.group(1))
    except NoSuchElementException:
        pass
    return out


def parse_category_ratings(driver):
    mapping = {
        "cleanliness": "cleanliness_rating",
        "accuracy": "accuracy_rating",
        "check-in": "check_in_rating",
        "communication": "communication_rating",
        "location": "location_rating",
        "value": "value_rating",
    }
    out = {v: None for v in mapping.values()}

    try:
        reviews_h2 = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located(
                (By.XPATH, "//h2[contains(., 'Rated') and contains(., 'reviews')]")))
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", reviews_h2)
        time.sleep(2)
    except TimeoutException:
        pass

    html = driver.page_source
    for m in re.finditer(r"Rated\s+([\d.]+)\s+out of 5 stars for\s+([A-Za-z\-]+)", html):
        key = mapping.get(m.group(2).strip().lower())
        if key and out[key] is None:
            out[key] = float(m.group(1))
    return out


def parse_superhost(driver):
    try:
        driver.find_element(
            By.XPATH, "//*[contains(@aria-label,'is a superhost') or "
                      "normalize-space(text())='Superhost']")
        return True
    except NoSuchElementException:
        return False


def parse_price_and_nights(driver):
    """total_price من زر السعر: '$3,306 for 5 nights, originally $4,246'"""
    total_price, nights = None, None
    try:
        btn = driver.find_element(
            By.XPATH, "//aside//button[contains(., 'for') and contains(., 'night')]")
        txt = btn.get_attribute("textContent")
        m = re.search(r"\$?([\d,]+(?:\.\d+)?)\s*for\s*(\d+)\s*night", txt)
        if m:
            total_price = float(m.group(1).replace(",", ""))
            nights = int(m.group(2))
    except NoSuchElementException:
        pass

    if nights is None:
        try:
            el = driver.find_element(By.XPATH, "//*[starts-with(normalize-space(.), 'for ') "
                                               "and contains(., 'nights')]")
            m = re.search(r"(\d+)\s*night", el.text)
            if m:
                nights = int(m.group(1))
        except NoSuchElementException:
            pass

    if total_price is None:
        try:
            el = driver.find_element(
                By.XPATH, "//aside//span[starts-with(normalize-space(.), '$')]")
            m = re.search(r"([\d,]+(?:\.\d+)?)", el.text)
            if m:
                total_price = float(m.group(1).replace(",", ""))
        except NoSuchElementException:
            pass

    return total_price, nights


def _find_key(obj, key):
    if isinstance(obj, dict):
        if key in obj and obj[key]:
            return obj[key]
        for v in obj.values():
            r = _find_key(v, key)
            if r:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = _find_key(v, key)
            if r:
                return r
    return None


def parse_amenities(driver):
    amenities = []

    scripts = driver.find_elements(By.CSS_SELECTOR, 'script[type="application/json"]')
    for s in scripts:
        raw = s.get_attribute("textContent") or ""
        if "seeAllAmenitiesGroups" not in raw:
            continue
        try:
            data = json.loads(raw)
        except Exception:
            continue

        groups = _find_key(data, "seeAllAmenitiesGroups") or []
        for g in groups:
            for a in g.get("amenities", []):
                if a.get("available") is True:
                    title = (a.get("title") or "").strip()
                    if title and title not in amenities:
                        amenities.append(title)
        if amenities:
            return amenities

    html = driver.page_source
    pattern = r'"__typename":"AmenityItem","id":"pdp_v3_[^"]+","available":true,"title":"([^"]+)"'
    for m in re.finditer(pattern, html):
        title = m.group(1).encode().decode("unicode_escape", errors="ignore") \
            if "\\u" in m.group(1) else m.group(1)
        if title not in amenities:
            amenities.append(title)

    return amenities





def parse_category_ratings_json(driver):
    mapping = {
        "CLEANLINESS": "cleanliness_rating", "ACCURACY": "accuracy_rating",
        "CHECKIN": "check_in_rating", "COMMUNICATION": "communication_rating",
        "LOCATION": "location_rating", "VALUE": "value_rating",
    }
    out = {v: None for v in mapping.values()}
    for s in driver.find_elements(By.CSS_SELECTOR, 'script[type="application/json"]'):
        raw = s.get_attribute("textContent") or ""
        if "categoryRatings" not in raw:
            continue
        try:
            ratings = _find_key(json.loads(raw), "categoryRatings") or []
        except Exception:
            continue
        for r in ratings:
            key = mapping.get(r.get("categoryType"))
            if key:
                out[key] = float(r["localizedRating"])
        break
    return out

def scrape_listing(driver, listing_url, city=None):
    data = {
        "city": city,
        "listing_url": listing_url,
        "listing_title": None,
        "guests": None, "bedrooms": None, "beds": None, "baths": None,
        "rating": None, "reviews": None,
        "cleanliness_rating": None, "accuracy_rating": None,
        "check_in_rating": None, "communication_rating": None,
        "location_rating": None, "value_rating": None,
        "lat": None, "lng": None,
        "room_type": None, "is_superhost": None, "property_type": None,
        "total_price": None, "nights": None,
        "amenities": [],
    }

    full_url = f"{listing_url}?adults=1&check_in={CHECK_IN}&check_out={CHECK_OUT}"
    driver.get(full_url)

    try:
        WebDriverWait(driver, 30).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1")))
    except TimeoutException:
        print("Timeout:", listing_url)
        return data

    time.sleep(3)
    close_popups(driver)

    data["listing_title"] = safe_text(driver, By.TAG_NAME, "h1")

    ld = get_json_ld(driver)
    data["lat"] = ld.get("lat")
    data["lng"] = ld.get("lng")
    data["rating"] = float(ld["rating"]) if ld.get("rating") else None
    data["reviews"] = int(ld["reviews"]) if ld.get("reviews") else None
    if not data["listing_title"]:
        data["listing_title"] = ld.get("title")

    data.update(parse_overview(driver))
    data["is_superhost"] = parse_superhost(driver)
    data["total_price"], data["nights"] = parse_price_and_nights(driver)

    scroll_through_page(driver)

    ratings = parse_category_ratings_json(driver)
    if all(v is None for v in ratings.values()):
        ratings = parse_category_ratings(driver)
    data.update(ratings)

    data["amenities"] = parse_amenities(driver)

    return data

In [ ]:
# =========================
# Storage
# =========================
city_count = 0
CHECK_IN = "2026-11-01"
CHECK_OUT = "2026-11-06"
target = 50
try:

    for city in cities:

        city_data = []

        try:

            url = (
                f"https://www.airbnb.com/s/{city}/homes?"
                "refinement_paths%5B%5D=%2Fhomes"
                f"&checkin={CHECK_IN}"
                f"&checkout={CHECK_OUT}"
                "&min_bedrooms=2"
                "&selected_filter_order%5B%5D=min_bedrooms%3A2"
                "&selected_filter_order%5B%5D=room_types%3AEntire%20home%2Fapt"
                "&update_selected_filters=true"
                f"&query={city}"
                "&search_mode=regular_search"
                "&room_types%5B%5D=Entire%20home%2Fapt"
                "&pagination_search=true"
            )
            print(f"\nCity: {city}")

            # =========================
            # Collect listing URLs
            # =========================

            listing_urls = []
            seen_urls = set()
            visited_pages = set()



            try:

                driver.get(url)

            except Exception as e:

                print(
                    f"Could not open city page for {city}:",
                    e
                )

                continue

            try:
                time.sleep(8)
            except Exception:
                pass

            try:
                print("Page title:", driver.title)
            except Exception:
                print("Page title: None")

            try:
                print("Current URL:", driver.current_url)
            except Exception:
                print("Current URL: None")

            try:
                print("Page source length:", len(driver.page_source))
            except Exception:
                print("Page source length: None")

            # =====================================================
            # PAGINATION / COLLECT LISTINGS
            # =====================================================

            try:

                while len(listing_urls) < target:

                    # ---------------------------------------------
                    # Current URL
                    # ---------------------------------------------

                    try:

                        current_url = driver.current_url

                    except Exception:

                        current_url = None

                    if not current_url:

                        print(
                            "Could not get current URL. Stopping pagination."
                        )

                        break

                    # ---------------------------------------------
                    # Prevent duplicate pages
                    # ---------------------------------------------

                    try:

                        if current_url in visited_pages:

                            print(
                                "Page already visited. Stopping."
                            )

                            break

                        visited_pages.add(current_url)

                    except Exception:

                        pass

                    # ---------------------------------------------
                    # Wait for listing cards
                    # ---------------------------------------------

                    try:

                        wait.until(
                            EC.presence_of_element_located(
                                (
                                    By.CSS_SELECTOR,
                                    '[data-testid="card-container"]'
                                )
                            )
                        )

                    except Exception as e:

                        print(
                            "Listing cards were not found:",
                            e
                        )

                        break

                    try:
                        time.sleep(3)
                    except Exception:
                        pass

                    # ---------------------------------------------
                    # Get cards
                    # ---------------------------------------------

                    try:

                        cards = driver.find_elements(
                            By.CSS_SELECTOR,
                            '[data-testid="card-container"]'
                        )

                    except Exception as e:

                        print(
                            "Could not get listing cards:",
                            e
                        )

                        cards = []

                    print(
                        "\nCards found:",
                        len(cards)
                    )

                    # ---------------------------------------------
                    # Extract listing URLs
                    # ---------------------------------------------

                    for card in cards:

                        try:

                            links = card.find_elements(
                                By.CSS_SELECTOR,
                                'a[href*="/rooms/"]'
                            )

                        except Exception:

                            links = []

                        for link in links:

                            try:

                                href = link.get_attribute("href")

                            except Exception:

                                href = None

                            if href:

                                try:

                                    listing_url = (
                                        urljoin(
                                            "https://www.airbnb.com",
                                            href
                                        )
                                        .split("?")[0]
                                    )

                                except Exception:

                                    listing_url = None

                                if listing_url:

                                    try:

                                        if (
                                            listing_url
                                            not in seen_urls
                                        ):

                                            seen_urls.add(
                                                listing_url
                                            )

                                            listing_urls.append(
                                                listing_url
                                            )

                                    except Exception:

                                        pass

                                break

                        if len(listing_urls) >= target:

                            break

                    print(
                        "Total unique listings:",
                        len(listing_urls)
                    )

                    if len(listing_urls) >= target:

                        break

                    # =================================================
                    # Find next page
                    # =================================================

                    next_url = None

                    # ---------------------------------------------
                    # Pagination links
                    # ---------------------------------------------

                    try:

                        pagination_links = driver.find_elements(
                            By.CSS_SELECTOR,
                            'nav[aria-label="Search results pagination"] a[href]'
                        )

                    except Exception as e:

                        print(
                            "Could not find pagination links:",
                            e
                        )

                        pagination_links = []

                    print(
                        "Pagination links found:",
                        len(pagination_links)
                    )

                    # ---------------------------------------------
                    # Next button
                    # ---------------------------------------------

                    for link in pagination_links:

                        try:

                            aria_label = (
                                link.get_attribute("aria-label")
                                or ""
                            ).strip().lower()

                        except Exception:

                            aria_label = ""

                        if aria_label == "next":

                            try:

                                next_url = link.get_attribute(
                                    "href"
                                )

                            except Exception:

                                next_url = None

                            if next_url:

                                break

                    # ---------------------------------------------
                    # Numbered page fallback
                    # ---------------------------------------------

                    if not next_url:

                        for link in pagination_links:

                            try:

                                aria_label = (
                                    link.get_attribute(
                                        "aria-label"
                                    )
                                    or ""
                                ).strip()

                            except Exception:

                                aria_label = ""

                            if aria_label.lower().startswith(
                                "page "
                            ):

                                try:

                                    href = link.get_attribute(
                                        "href"
                                    )

                                except Exception:

                                    href = None

                                if (
                                    href
                                    and href not in visited_pages
                                ):

                                    next_url = href
                                    break

                    # ---------------------------------------------
                    # Cursor fallback
                    # ---------------------------------------------

                    if not next_url:

                        try:

                            all_links = driver.find_elements(
                                By.CSS_SELECTOR,
                                "a[href]"
                            )

                        except Exception:

                            all_links = []

                        for link in all_links:

                            try:

                                href = (
                                    link.get_attribute(
                                        "href"
                                    )
                                    or ""
                                )

                            except Exception:

                                href = ""

                            try:

                                aria_label = (
                                    link.get_attribute(
                                        "aria-label"
                                    )
                                    or ""
                                ).strip().lower()

                            except Exception:

                                aria_label = ""

                            if (
                                "cursor=" in href
                                and (
                                    aria_label == "next"
                                    or "page 2 of" in aria_label
                                )
                            ):

                                next_url = href
                                break

                    # ---------------------------------------------
                    # No next page
                    # ---------------------------------------------

                    if not next_url:

                        print(
                            "No next-page link found."
                        )

                        break

                    # ---------------------------------------------
                    # Normalize next URL
                    # ---------------------------------------------

                    try:

                        next_url = urljoin(
                            "https://www.airbnb.com",
                            next_url
                        )

                    except Exception:

                        next_url = None

                    if not next_url:

                        break

                    # ---------------------------------------------
                    # Prevent duplicate next page
                    # ---------------------------------------------

                    try:

                        if (
                            next_url in visited_pages
                            or next_url == current_url
                        ):

                            print(
                                "Next page already visited."
                            )

                            break

                    except Exception:

                        pass

                    print(
                        "Moving to next page..."
                    )

                    try:

                        driver.get(next_url)

                    except Exception as e:

                        print(
                            "Could not open next page:",
                            e
                        )

                        break

                    try:
                        time.sleep(5)
                    except Exception:
                        pass

            except Exception as e:

                print(
                    f"Pagination error for {city}:",
                    e
                )

            print(
                f"\nTotal listing URLs collected for {city}: "
                f"{len(listing_urls)}"
            )

            # =====================================================
            # SCRAPE EACH LISTING
            # =====================================================

            for i, listing_url in enumerate(
                listing_urls,
                start=1
            ):

                try:

                    print(
                        f"\n{'=' * 60}"
                    )

                    print(
                        f"Listing {i}/{len(listing_urls)}"
                    )

                    print(
                        "URL:",
                        listing_url
                    )

                    # =================================================
                    # OPEN LISTING
                    # =================================================

                    try:
                        row = scrape_listing(driver, listing_url, city=city)
                    except Exception as e:

                        print(
                            "Could not open listing:",
                            e
                        )

                        continue

                    # =================================================
                    # SAVE ONE COMPLETE ROW
                    # =================================================
                    try:
                        row = scrape_listing(driver, listing_url, city=city)
                    except Exception as e:
                        print("Error scraping listing:", repr(e))
                        traceback.print_exc()
                        continue

                    city_data.append(row)
                    city_count += 1
                    print("Saved. Amenities:", len(row.get("amenities", [])),
                          "| price:", row.get("total_price"),
                          "| cleanliness:", row.get("cleanliness_rating"))

                    time.sleep(random.uniform(3, 6))

                except Exception as e:

                    # =================================================
                    # LAST SAFETY NET
                    # =================================================

                    print(
                        f"Unexpected error in listing {i}:",
                        e
                    )

                    try:

                        city_data.append(row)

                        city_count += 1

                        print(
                            "Partial listing saved."
                        )

                    except Exception as save_error:

                        print(
                            "Could not save partial listing:",
                            save_error
                        )

        except Exception as e:

            print(
                f"Error processing city {city}:",
                e
            )

        # =====================================================
        # CREATE DATAFRAME AND SAVE CSV PER CITY
        # =====================================================
        try:
            if city_data:
                df_city = pd.DataFrame(city_data)
                csv_filename = f"airbnb_{city.lower()}_listings_comp.csv"
                df_city.to_csv(
                    csv_filename,
                    index=False,
                    encoding="utf-8-sig"
                )
                print(f"Total non nan df{len(df_city.dropna())}")
                print(f"\nSaved CSV for {city}: {csv_filename} ({len(df_city)} records)")
                files.download(csv_filename)
            else:
                print(f"\nNo data collected for {city}, skipping CSV creation.")
        except Exception as e:
            print(f"Could not create or save CSV for {city}:", e)

# =====================================================
# FINALLY
# =====================================================

finally:

    try:

        driver.quit()

        print(
            "\nBrowser closed."
        )

    except Exception as e:

        print(
            "\nCould not close browser:",
            e
        )

try:
    print(
        "\nTotal listings visited across all cities:",
        city_count
    )
except Exception:
    pass

In [ ]:
# Print the transposed dataframe containing the scraped data
try:
    print(df_city.T.to_string(header=False))
except NameError as e:
    print(f"Variable error: {e}. Please run cell rFodMhPKYoh3 first to populate df_city.")

city                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               Barcelona                                                                                                                                                                                                                                                        Barc